In [11]:
import pandas as pd 
import sqlite3
import prettytable
from prettytable import TableStyle
prettytable.DEFAULT = TableStyle.DEFAULT  

conn = sqlite3.connect('polling.db')
curs = conn.cursor()

# sqlite_web polling.db

In [1]:
%load_ext sql
%sql sqlite:///polling.db

In [78]:
def Create_Tables():

    with sqlite3.connect('polling.db') as conn:
        
        curs = conn.cursor()

        curs.execute(
            '''
            CREATE TABLE IF NOT EXISTS counties (
                county_id INTEGER PRIMARY KEY AUTOINCREMENT,
                county_name TEXT,
                fips_code TEXT,
                UNIQUE (county_name, fips_code)
            )
            '''
        )

        curs.execute(
            '''
            CREATE TABLE IF NOT EXISTS jurisdictions (
                jurisdiction_id INTEGER PRIMARY KEY AUTOINCREMENT,
                county_id INTEGER,
                jurisdiction_name TEXT,
                clerk_name TEXT,
                clerk_email TEXT,
                jurisdiction_type TEXT,
                FOREIGN KEY (county_id) REFERENCES counties(county_id),
                UNIQUE (county_id, jurisdiction_name)
            )
            '''
        )

        curs.execute(
            '''
            CREATE TABLE IF NOT EXISTS locations (
                location_id INTEGER PRIMARY KEY AUTOINCREMENT,
                jurisdiction_id INTEGER,
                location_name TEXT,
                address TEXT,
                city TEXT,
                zip_code TEXT,
                latitude REAL,
                longitude REAL,
                FOREIGN KEY (jurisdiction_id) REFERENCES jurisdictions(jurisdiction_id),
                UNIQUE (jurisdiction_id, location_name, address)
            )
            '''
        )
        
        curs.execute(
            '''
            CREATE TABLE IF NOT EXISTS elections (
                election_id INTEGER PRIMARY KEY AUTOINCREMENT,
                election_year INTEGER,
                election_date TEXT,
                election_type TEXT,
                notes TEXT,
                UNIQUE (election_year, election_date, election_type)
            )
            '''
        )

        curs.execute(
            '''
            CREATE TABLE IF NOT EXISTS election_usage (
                usage_id INTEGER PRIMARY KEY AUTOINCREMENT,
                election_id INTEGER,
                location_id INTEGER,
                location_function TEXT,
                open_date TEXT,
                close_date TEXT,
                hours TEXT,
                notes TEXT,
                FOREIGN KEY (election_id) REFERENCES elections(election_id),
                FOREIGN KEY (location_id) REFERENCES locations(location_id),
                UNIQUE (election_id, location_id, location_function)
            )
            '''
        )

In [79]:
Create_Tables()

In [6]:
%%sql 
SELECT * FROM sqlite_master WHERE type='table';

 * sqlite:///polling.db
Done.


type,name,tbl_name,rootpage,sql
table,counties,counties,2,"CREATE TABLE counties ( county_id INTEGER PRIMARY KEY AUTOINCREMENT, county_name TEXT, fips_code TEXT, UNIQUE (county_name, fips_code) )"
table,sqlite_sequence,sqlite_sequence,4,"CREATE TABLE sqlite_sequence(name,seq)"
table,jurisdictions,jurisdictions,5,"CREATE TABLE jurisdictions ( jurisdiction_id INTEGER PRIMARY KEY AUTOINCREMENT, county_id INTEGER, jurisdiction_name TEXT, clerk_name TEXT, clerk_email TEXT, jurisdiction_type TEXT, FOREIGN KEY (county_id) REFERENCES counties(county_id), UNIQUE (county_id, jurisdiction_name) )"
table,elections,elections,9,"CREATE TABLE elections ( election_id INTEGER PRIMARY KEY AUTOINCREMENT, election_year INTEGER, election_date TEXT, election_type TEXT, notes TEXT, UNIQUE (election_year, election_date, election_type) )"
table,election_usage,election_usage,11,"CREATE TABLE election_usage ( usage_id INTEGER PRIMARY KEY AUTOINCREMENT, election_id INTEGER, location_id INTEGER, location_function TEXT, open_date TEXT, close_date TEXT, hours TEXT, notes TEXT, FOREIGN KEY (election_id) REFERENCES elections(election_id), FOREIGN KEY (location_id) REFERENCES locations(location_id), UNIQUE (election_id, location_id, location_function) )"
table,locations,locations,7,"CREATE TABLE locations ( location_id INTEGER PRIMARY KEY AUTOINCREMENT, jurisdiction_id INTEGER, location_name TEXT, address TEXT, city TEXT, zip_code TEXT, latitude REAL, longitude REAL, FOREIGN KEY (jurisdiction_id) REFERENCES jurisdictions(jurisdiction_id), UNIQUE (jurisdiction_id, location_name, address) )"


# County Loader

In [33]:
county_df = pd.read_csv('county.csv')
county_df.columns

Index(['FIPS', 'County_Name'], dtype='object')

In [ ]:
with sqlite3.connect('polling.db') as conn:

    curs = conn.cursor()

    for index, row in county_df.iterrows():
    
        try: 
            
            curs.execute(
                '''
                INSERT INTO counties (county_name, fips_code)
                VALUES (?, ?)
                ''',
                (row['County_Name'], row['FIPS'])
            )
        
        except sqlite3.IntegrityError:

            print(f"Skipped row {index}: UNIQUE constraint failed: {row['county_name']} ({row['FIPS']})")

# Jurisdiction Loader

In [24]:
jurisdiction_df = pd.read_csv('jurisdiction.csv')   
jurisdiction_df.columns

Index(['county_name', 'jurisdiction_name', 'clerk_name', 'clerk_email',
       'jurisdiction_type'],
      dtype='object')

In [25]:
with sqlite3.connect('polling.db') as conn:

    curs = conn.cursor()

    for index, row in jurisdiction_df.iterrows():

        try: 
            
            county_id = curs.execute("SELECT county_id FROM counties WHERE county_name = ?", (row['county_name'],)).fetchone()[0]

            if county_id is None:

                print(f"County not found for row {index}: {row['county_name']}")
            
                continue

            curs.execute(
                '''
                INSERT INTO jurisdictions (county_id, jurisdiction_name, clerk_name, clerk_email, jurisdiction_type)
                VALUES (?, ?, ?, ?, ?)
                ''',
                (county_id, row['jurisdiction_name'], row['clerk_name'], row['clerk_email'], row['jurisdiction_type'])
            )

        except sqlite3.IntegrityError:
            
            print(f"Skipped row {index}: UNIQUE constraint failed: {row['jurisdiction_name']} ({row['county_name']})")

# Locations Loader

In [28]:
locations_df = pd.read_csv('locations.csv')
locations_df.columns

Index(['county_name', 'jurisdiction_name', 'location_name', 'address', 'city',
       'zip_code', 'latitude', 'longitude'],
      dtype='object')

In [30]:
with sqlite3.connect('polling.db') as conn:
        
    curs = conn.cursor()    

    for index, row in locations_df.iterrows():

        try: 
            
            county_id = curs.execute("SELECT county_id FROM counties WHERE county_name = ?", (row['county_name'],)).fetchone()[0]
            jurisdiction_id = curs.execute(f"SELECT jurisdiction_id FROM jurisdictions WHERE jurisdiction_name = ? AND county_id = {county_id}", (row['jurisdiction_name'],)).fetchone()[0]

            if county_id is None:

                print(f"Skipped row {index}: No county found for {row['county_name']}")

                continue

            if jurisdiction_id is None:

                print(f"Skipped row {index}: No jurisdiction found for {row['jurisdiction_name']} in county {row['county_name']}")
                
                continue
        
            curs.execute(
                '''
                INSERT INTO locations (jurisdiction_id, location_name, address, city, zip_code, latitude, longitude)   
                VALUES (?, ?, ?, ?, ?, ?, ?)
                ''',
                (jurisdiction_id, row['location_name'], row['address'], row['city'], row['zip_code'], row['latitude'], row['longitude'])
            )

        except sqlite3.IntegrityError:

            print(f"Skipped row {index}: UNIQUE constraint failed: {row['jurisdiction_name']} ({row['location_name']}, {row['address']})")

# Elections Loader

In [31]:
elections_df = pd.read_csv('elections.csv')
elections_df.columns

Index(['election_year', 'election_date', 'election_type', 'notes'], dtype='object')

In [32]:
with sqlite3.connect('polling.db') as conn:

    curs = conn.cursor()

    for index, row in elections_df.iterrows():

        try:

            curs.execute(
                '''
                INSERT INTO elections (election_year, election_date, election_type, notes)   
                VALUES (?, ?, ?, ?)
                ''',
                (row['election_year'], row['election_date'], row['election_type'], row['notes'])
            )

        except sqlite3.IntegrityError:

            print(
                f"Skipped row {index}: UNIQUE constraint failed: "
                f"{row['election_year']} ({row['election_date']}, {row['election_type']})"
            )

Skipped row 0: UNIQUE constraint failed: 2024 (2024-11-05, General)
Skipped row 1: UNIQUE constraint failed: 2023 (2023-05-02, Special)
Skipped row 2: UNIQUE constraint failed: 2022 (2022-08-09, Primary)
Skipped row 3: UNIQUE constraint failed: 2020 (2020-03-10, Primary)
Skipped row 4: UNIQUE constraint failed: 2019 (2019-11-05, Local)


# Elections Usage Loader

In [33]:
elections_usage_df = pd.read_csv('election_usage.csv')
elections_usage_df.columns

Index(['location_name', 'address', 'election_type', 'election_date',
       'location_function', 'open_date', 'close_date', 'hours', 'notes'],
      dtype='object')

In [34]:
with sqlite3.connect('polling.db') as conn:
    
    curs = conn.cursor()

    for index, row in elections_usage_df.iterrows():

        try: 
            
            location_id = curs.execute("SELECT location_id FROM locations WHERE location_name = ? AND address = ?", (row['location_name'], row['address'])).fetchone()[0]
            election_id = curs.execute("SELECT election_id FROM elections WHERE election_type = ? AND election_date = ?", (row['election_type'], row['election_date'])).fetchone()[0]

            if location_id is None:

                print(f"Skipped row {index}: location not found {row['location_name']} @ {row['address']}")

                continue

            if election_id is None:

                print(f"Skipped row {index}: election not found {row['election_type']} on {row['election_date']}")

                continue

            curs.execute(
                '''
                INSERT INTO election_usage (election_id, location_id, location_function, open_date, close_date, hours, notes)   
                VALUES (?, ?, ?, ?, ?, ?, ?)
                ''',
                (election_id, location_id, row['location_function'], row['open_date'], row['close_date'], row['hours'], row['notes'])
            )

        except sqlite3.IntegrityError:

            print(f"Skipped row {index}: UNIQUE constraint failed: [{row['election_type']}, {row['election_date']}], [{row['location_name']}, {row['address']}], {row['location_function']})")